# Seizure Simulation in Clustered Networks — NEURON

A single-compartment **HH + A-current (`kA`) + dynamic [K+]o (`kdyn`)** clustered
network. Under weak Poisson drive it produces discrete **network bursts**; impairing
glial K+ clearance (raising **`tau_k`**) drives a **seizure**: firing accumulates
[K+]o, E_K depolarizes, and positive feedback pushes the network into an ictal state.

This notebook will:

1. build a log-normal clustered topology,
2. **verify the network bursts** in the normal state (tuned defaults),
3. compare **normal vs seizure** (`tau_k`) with a [K+]o overlay,
4. trace a **seizure dose-response** (peak [K+]o vs `tau_k`),
5. **save an inference-ready dataset**, and
6. run the **CCG + learned-LIF inference** on it (AUC / FDR).

> **Prerequisite:** compile the mechanisms once (`kA`, `kdyn`, `DepSyn`):
> ```bash
> cd neuron_simulation && nrnivmodl mechanisms
> ```

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = os.path.abspath('..')
for p in (REPO_ROOT, os.path.join(REPO_ROOT, 'inference')):
    if p not in sys.path:
        sys.path.insert(0, p)

from neuron_simulation import (
    topology, build_network, run_simulation, states, analysis, plotting, workflows,
)

# The build_network defaults are the tuned, validated realistic regime:
#   AMPA+NMDA synapse (exc_tau=5 ms AMPA, tau_nmda=150 ms, nmda_ratio=2.5),
#   rodent short-term depression (d=0.3, tau_d=500 ms), inhibition-stabilized
#   drive (exc_weight_scale=2.5, inh_weight_scale=5.0, noise_rate=18 Hz), and a
#   two-timescale adaptation (fast SFA tau=300 ms + slow AHP tau=2.5 s). This
#   gives ~2 spikes/burst, ~90 ms bursts, IBI ~0.8 s, and uniform inter-burst
#   firing. Seizure = impaired K+ clearance (tau_k up) + reduced slow sAHP.
CONFIG = {
    'topology': dict(
        num_clusters=20, neurons_per_cluster_range=(4, 40),
        inhibitory_probability=0.2, space_size=15.0,
        within_cluster_prob=0.2, between_cluster_prob=0.15,
        target_density=None,          # density emerges from the connectivity rules
        ln_sigma=1.0, seed=1,
    ),
    'build': {},          # empty -> use the tuned realistic defaults
    'sim': dict(dt=0.05, discard_transient_ms=1000.0),
}

# ==========================================================================
# PINNED SETTING: "blanket" -- cell-type-specific local inhibitory blanket.
# Reproduces raster_blanket_tuned.png (N=396, seed 1): density ~2.0%,
# ~1.3 Hz bursts, ~98% participation, ~135 ms. Inhibition is local-only, so
# between-cluster edges are a ~purely excitatory backbone (raised to 0.05).
# ==========================================================================
CONFIG['topology'].update(dict(
    num_clusters=20, neurons_per_cluster_range=(4, 40), inhibitory_probability=0.2,
    space_size=15.0, cell_type_specific=True,
    p_ee_within=0.15, p_ee_between=0.05,          # E backbone raised from 0.02
    p_ei_within=0.30, p_ei_between=0.01,
    p_ie_within=0.70, p_ii_within=0.50,           # local blanket
    ln_sigma=0.5, target_density=None, seed=1,
))
CONFIG['build'].update(dict(
    synapse_model='ampa_nmda', exc_tau=5.0, tau_nmda=150.0, nmda_ratio=2.5,
    depression_d=0.3, tau_d=500.0,
    exc_weight_scale=6.0, inh_weight_scale=2.5,   # strong E, lowered I
    noise_rate=18.0, noise_weight=0.002,
    adapt=True,
    sahp_ainc_fast=0.005, sahp_tau_fast=300.0,
    sahp_ainc_slow=0.0015, sahp_tau_slow=2500.0,
    delay_per_distance=2.0,
))
print('config ready (pinned setting: blanket)')

## 1. Build the topology (log-normal — preferred)

Continuous heavy-tailed degree distribution (no bimodal gap) at ~3–4% density.
Use `topology.build_topology(...)` for the discrete-hub variant.

In [ ]:
topo = topology.build_topology_lognormal(**CONFIG['topology'])
N = topo['n_neurons']

# One-call topology view: prints the inter/between-cluster connection stats
# (density, within/between rates, ratio, hub share) and returns the 4-panel
# overview (spatial layout, hub fan-out, cluster-sorted connection matrix,
# out-degree distribution). Supersedes the separate map + degree plots.
stats, fig = plotting.topology_report(topo)
plt.show()

## 2. Verify the network bursts (normal state)

**Gate:** discrete, high-participation bursts with a sparse inter-burst baseline.
A network burst = **> 80% of neurons firing within the event window** (post burn-in).

In [ ]:
normal = workflows.run_single_state(
    topo, state=states.normal_state(), build_kwargs=CONFIG['build'],
    duration=10000.0, record_ko=True, **CONFIG['sim'])
stats = normal['burst_stats']
rate = analysis.firing_rate_summary(normal['spike_data'], 10000.0, burn_in_ms=0.0)['mean_rate_hz']
print('mean rate (Hz):', round(rate, 2), '| burst stats:', stats)
print('[K+]o mean range (mM):', round(float(normal['ko_data']['mean_ko'].min()), 2),
      '-', round(float(normal['ko_data']['mean_ko'].max()), 2))
assert stats['n_bursts'] >= 1 and not stats['merged'], 'network did not burst'
assert stats['mean_participation'] > 0.8, 'bursts below 80% participation'
print('OK: discrete network bursts with >80% participation')

# (1) cluster-sorted raster with [K+]o
fig = plotting.plot_raster_with_ko(
    normal['spike_data'], N, 10000.0, normal['ko_data'],
    is_inhibitory=topo['neuron_is_inhibitory'],
    cluster_assignments=topo['cluster_assignments'], title='Normal (cluster-sorted)')
plt.show()

# (2) SAME run, neuron rows randomized -- if the burst synchrony is genuinely
# network-wide the vertical stripes survive the shuffle; if it were an artifact
# of grouping clusters by index they would smear out.
fig = plotting.plot_raster_with_ko(
    normal['spike_data'], N, 10000.0, normal['ko_data'],
    is_inhibitory=topo['neuron_is_inhibitory'],
    cluster_assignments=topo['cluster_assignments'],
    title='Normal (randomized neuron rows)', randomize_rows=True)
plt.show()

## 3. Normal vs seizure (impaired K+ clearance)

Seizure = large `tau_k` (impaired glial buffering). [K+]o should climb from ~4 mM
into the ictal range (~8–12 mM) with elevated, less-discrete firing.

In [ ]:
seizure = workflows.run_single_state(
    topo, state=states.seizure_state(1.0), build_kwargs=CONFIG['build'],
    duration=10000.0, record_ko=True, **CONFIG['sim'])
print('normal  [K+]o max (mM):', round(float(normal['ko_data']['mean_ko'].max()), 2))
print('seizure [K+]o max (mM):', round(float(seizure['ko_data']['mean_ko'].max()), 2))

# (1) cluster-sorted
fig = plotting.plot_raster_with_ko(
    seizure['spike_data'], N, 10000.0, seizure['ko_data'],
    is_inhibitory=topo['neuron_is_inhibitory'],
    cluster_assignments=topo['cluster_assignments'], title='Seizure severity 1.0 (cluster-sorted)')
plt.show()

# (2) randomized neuron rows
fig = plotting.plot_raster_with_ko(
    seizure['spike_data'], N, 10000.0, seizure['ko_data'],
    is_inhibitory=topo['neuron_is_inhibitory'],
    cluster_assignments=topo['cluster_assignments'],
    title='Seizure severity 1.0 (randomized neuron rows)', randomize_rows=True)
plt.show()

## 4. Seizure dose-response: peak [K+]o vs `tau_k`

Sweep the clearance time constant from healthy to impaired and watch [K+]o rise
into the ictal range.

In [ ]:
sweep = states.seizure_dose_response(n_points=5)
tau_ks, ko_peaks, rates = [], [], []
for s in sweep:
    res = workflows.run_single_state(topo, state=s, build_kwargs=CONFIG['build'],
                                     duration=8000.0, record_ko=True, **CONFIG['sim'])
    tau_ks.append(s['tau_k'])
    ko_peaks.append(float(res['ko_data']['mean_ko'].max()))
    rates.append(analysis.firing_rate_summary(res['spike_data'], 8000.0, burn_in_ms=0.0)['mean_rate_hz'])
    print(f"tau_k={s['tau_k']:.0f} ms -> peak [K+]o {ko_peaks[-1]:.1f} mM, rate {rates[-1]:.1f} Hz")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(tau_ks, ko_peaks, 'o-', color='#c0392b')
ax.axhline(4.0, color='k', ls=':', lw=1, label='rest (4 mM)')
ax.set_xlabel('tau_k (ms)  [larger = more impaired clearance]')
ax.set_ylabel('peak mean [K+]o (mM)')
ax.set_title('Seizure dose-response: [K+]o accumulation vs K+ clearance')
ax.legend(fontsize=8); plt.show()

## 5. Generate an inference-ready dataset (normal state)

Independent recordings (same wired network, different Random123 noise streams per
recording) saved in the exact LIF session layout.

In [ ]:
meta, session_dir = workflows.generate_dataset(
    n_recordings=3, recording_duration=15000.0,
    topology_kind='lognormal', topology_kwargs=CONFIG['topology'],
    build_kwargs=CONFIG['build'], state=states.normal_state(),
    record_voltage=False, save_dir='NEURON data', **CONFIG['sim'])
print('session saved to:', session_dir)
print('per-recording spikes (should differ):', [r['num_spikes'] for r in meta['recordings']])

## 6. Validate the format and run inference

Assert the saved session matches the inference contract, then run the **CCG baseline
+ learned-LIF** model and report **AUC / FDR** against the ground-truth wiring.

In [ ]:
import adapter  # inference/adapter.py
adapter.validate_session_format(session_dir)
summary = adapter.run_inference(
    session_dir, run_learned=True, run_ccg=True,
    learned_params={'n_epochs': 20, 'K': 60})
print(summary)

## Notes & caveats

- **Seizure knob is `tau_k` (K+ clearance)**, not `gbar_kA`. The reduced-A-current
  route (`states.gbar_block_state`) is deprecated to a phenomenological knob — its
  dramatic effect was specific to the dense discrete-hub topology.
- **Burst rate is fast** (~1.3 Hz) vs real cultures (~0.03 Hz).
- **Squid HH kinetics** at 6.3 °C; set `celsius=34` in `CONFIG['build']` for a
  faster mammalian-like variant. `kdyn` is a lumped single-pool [K+]o caricature.
- The **ground truth is the exact wired graph** (`connections`); inference sees
  steady-state data (the first ~1 s transient is dropped at save time).

## Parameter tuning guide

All knobs below are `build_kwargs` (passed to `build_network`) unless noted. The
defaults already give the realistic regime; change these to move the operating
point. Effects are coupled -- see the trade-offs at the end.

**Network dynamics**

| Want to change | Knob | Direction |
|---|---|---|
| Burst *rate* (slower/faster) | `sahp_ainc_slow`, `sahp_tau_slow` | higher -> slower bursts (longer IBI). Default 0.2 Hz. |
| *Spikes per burst* (thinner) | `sahp_ainc_fast` | higher -> fewer spikes/burst. Default ~2. |
| *Inter-burst background* firing | `noise_weight`, `noise_rate` | higher -> more background (fills gaps) |
| Burst *duration* | `nmda_ratio`, `delay_per_distance` | higher -> longer (reverberation / propagation) |
| Overall excitability | `exc_weight_scale`, `inh_weight_scale` | exc up / inh down -> more excitable |

**Topology** (`topology.build_topology_lognormal(...)` args)

| Want to change | Knob | Notes |
|---|---|---|
| Clustering strength | `within_cluster_prob` | *only works with* `target_density=None` |
| Overall density / in-degree | `target_density` | `None` = emerge from probs; a number = fixed density |
| Network size | `num_clusters`, `neurons_per_cluster_range` | e.g. (4, 40) for heterogeneous clusters |
| Spatial extent | `space_size` | scales the distance-dependent delays |

**Seizure** (`states.py` module constants)

| Want to change | Knob | Direction |
|---|---|---|
| Seizure firing/burst rate | `SEIZURE_SAHP_SLOW`, `SEIZURE_SAHP_FAST` | higher -> calmer (more adaptation retained) |
| Ictal [K+]o level | `SEIZURE_TAU_K` | higher -> more [K+]o accumulation |
| Per-run severity | `states.seizure_state(severity)` | higher severity -> higher `tau_k` -> stronger |

**Coupled trade-offs (can't optimize independently)**

- **Burst rate vs inter-burst firing:** a slower burst rate needs a stronger slow AHP, which also suppresses inter-burst firing. Slower bursts -> sparser background.
- **Spikes/burst vs burst duration:** fewer spikes/cell -> shorter bursts (duration is roughly spikes x intra-burst ISI), unless bursts propagate spatially.
- **Seizure firing vs ictal [K+]o:** firing *drives* [K+]o, so a calmer seizure (bounded firing) also gives a bounded [K+]o (~8 mM). For a higher [K+]o at bounded firing, raise `SEIZURE_TAU_K` / severity rather than cutting adaptation.
- **Uniform vs empty gaps:** uniform background needs strong enough independent noise; below a floor the recurrent network organizes gap firing into sparse mini-bursts.

**Backward compatibility:** the pre-tuning regime is still reachable via `build_network(synapse_model='depsyn', adapt=False, delay_per_distance=0.0, ...)` and `build_topology_lognormal(target_density=0.035, within_cluster_prob=0.55, ...)`.